In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import speech_recognition as sr
from pydub import AudioSegment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from wordcloud import WordCloud
from collections import Counter
import re

c:\Users\Jonny Villareal\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
#Datos de filtros
fecha_i = '20260501'
fecha_f = '20260531'

mes = 'mayo'

Notas

In [3]:
base_notas = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Rutas_puntos_críticos/Notas de orión.xlsx')

base_notas.head()

,Tipo Nota,Subtipo Nota,Motivo,Nota,Observacion
0,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Varado,Varado / Carrocería / 06-04-2026 / 07:51 / Z5...,ok
1,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Mecánica,FALLA MECÁNICA / Aceleración / 06-04-2026 / 1...,ok
2,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Técnica,FALLA TÉCNICA / Aceleración / 06-04-2026 / 11...,ok
3,NOVEDADES SIRCI,EQUIPAMIENTO A BORDO VEHÍCULOS,Sirci,SIRCI / Torniquete Bloqueado / ticket: # 2250...,ok
4,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,Desvio,Autorizacion de Desvio / ID 114568 / 06-04-202...,ok


In [4]:
base_notas['Nota'] = (
    base_notas['Nota']
    .astype(str)
    .str.replace('\xa0', ' ', regex=False)   
    .str.replace(r'\s+', ' ', regex=True)   
    .str.strip()                             
)

base_notas.head()

,Tipo Nota,Subtipo Nota,Motivo,Nota,Observacion
0,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Varado,Varado / Carrocería / 06-04-2026 / 07:51 / Z50...,ok
1,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Mecánica,FALLA MECÁNICA / Aceleración / 06-04-2026 / 11...,ok
2,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Técnica,FALLA TÉCNICA / Aceleración / 06-04-2026 / 11:...,ok
3,NOVEDADES SIRCI,EQUIPAMIENTO A BORDO VEHÍCULOS,Sirci,SIRCI / Torniquete Bloqueado / ticket: # 22507...,ok
4,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,Desvio,Autorizacion de Desvio / ID 114568 / 06-04-202...,ok


In [5]:
base_notas['cantidad_campos'] = base_notas['Nota'].str.split('/').apply(len)

base_notas.head()

,Tipo Nota,Subtipo Nota,Motivo,Nota,Observacion,cantidad_campos
0,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Varado,Varado / Carrocería / 06-04-2026 / 07:51 / Z50...,ok,14
1,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Mecánica,FALLA MECÁNICA / Aceleración / 06-04-2026 / 11...,ok,14
2,NOVEDADES VEHÍCULOS ZONALES,SEGUN MANUAL DE OPERACIONES,Técnica,FALLA TÉCNICA / Aceleración / 06-04-2026 / 11:...,ok,14
3,NOVEDADES SIRCI,EQUIPAMIENTO A BORDO VEHÍCULOS,Sirci,SIRCI / Torniquete Bloqueado / ticket: # 22507...,ok,15
4,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,Desvio,Autorizacion de Desvio / ID 114568 / 06-04-202...,ok,14


Paradas

In [6]:
#imprortar paradas

paradas = pd.read_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Paraderos_Zonales_del_SITP.csv')

paradas.head(3)

,X,Y,objectid,cenefa,zona_sitp,nombre,via,direccion_bandera,localidad,longitud,latitud,consecutivo_zona,tipo_m_s,consola,panel,audio,zonas_nuevas,globalid,shape
0,1.001502e+06,1.010205e+06,1,001A00,00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481,001,M,AC 100 - KR 54 (001A00),AC 100 - KR 54,Avenida Calle 100 Carrera 54,C,{1C0DBC4E-15BC-4BBE-BC16-5628077DBE2E},NaN
1,1.003505e+06,1.009719e+06,2,001A01,01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091,001,M,AC 100 - KR 13 (001A01),AC 100 - KR 13,Avenida Calle 100 Carrera 13,B,{60A22A44-AD56-4DF4-A3E6-6830B9FB0792},NaN
2,1.001238e+06,1.018098e+06,3,001A02,02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867,001,S,AV. Boyacá - AC 170 (001A02),AV. Boyacá - AC 170,Avenida Boyacá Avenida Calle 170,C,{97155274-E4D5-45A4-891F-2DCA19FF10D9},NaN


Notas de bitácora

In [7]:
# Ruta de la carpeta que contiene los archivos de actividad de bus zonal
ruta_carpeta = 'Z:/01 base_datos/08 varados FMS'

# Fechas de inicio y fin para el filtro
fecha_inicio = f'{fecha_i}'
fecha_fin = f'{fecha_f}'

# Lista para almacenar los DataFrames
dataframes = []

# Recorrer todos los archivos en la carpeta
for nombre_archivo in os.listdir(ruta_carpeta):
    if nombre_archivo.endswith('_varados.csv'):
        # Extraer la fecha del nombre del archivo
        fecha_archivo = nombre_archivo[:8]  
        
        # Convertir la fecha a un formato adecuado para comparación
        fecha_archivo_dt = pd.to_datetime(fecha_archivo, format='%Y%m%d', errors='coerce')

        # Verificar si la fecha está dentro del rango deseado
        if fecha_inicio <= fecha_archivo <= fecha_fin:
            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)
            # Leer el archivo CSV omitiendo la primera fila vacía
            df = pd.read_csv(ruta_archivo, encoding='latin', low_memory=False)
            dataframes.append(df)

# Verificar si se encontraron DataFrames
if dataframes:
    # Consolidar todos los DataFrames en uno solo
    notas = pd.concat(dataframes, ignore_index=True)

    # # Guardar el DataFrame consolidado en un nuevo archivo
    # desg_troncal.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Indicadores/Indicadores_python/desgl_troncal_ago24.csv', index=False)
else:
    print("No se encontraron archivos para consolidar.")
    
notas.head(3)

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO


In [8]:
notas['Observaciones'] = (
    notas['Observaciones']
    .astype(str)
    .str.replace('\xa0', ' ', regex=False)   
    .str.replace(r'\s+', ' ', regex=True)   
    .str.strip()                             
)

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO


In [9]:
notas['cantidad_campos'] = notas['Observaciones'].str.split('/').apply(len)

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14


In [10]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(tipo, sub, cant):
    
    filtro = (
        (base_notas['Tipo Nota'] == tipo)&
        (base_notas['Subtipo Nota'] == sub)&
        (base_notas['cantidad_campos'] == cant)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not base_notas.loc[filtro].empty:
        # Obtener el primer valor
        mds = base_notas.loc[filtro, 'Observacion'].iloc[0]
        return mds if not pd.isna(mds) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
notas['estado_nota'] = notas.apply(
    lambda row: calcular_turno(
        row['Tipo Nota'],
        row['Subtipo Nota'],
        row['cantidad_campos']
    ),
    axis=1
)

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos,estado_nota
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok


In [11]:
notas['estado_nota'] = (
    notas['estado_nota']
    .replace(['nan', 'NaN', 'NAN', ''], None)
    .fillna('No coincide')                     
)

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos,estado_nota
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok


In [12]:
# notas['Movil_limpio'] = (
#     notas['Observaciones']
#     .astype(str)
#     .str.upper()
#     .str.extract(r'Z\s*\|?\s*(\d+)\s*-\s*(\d+)')  # captura ambas partes
#     .apply(lambda x: f'Z{x[0]}{x[1]}' if pd.notna(x[0]) else pd.NA, axis=1)
# )

# notas.head()

In [13]:
col = notas['Observaciones'].astype(str).str.upper()

# 1️⃣ Ya tienes Z (se mantiene)
movil_z = col.str.extract(r'Z\s*\|?\s*(\d{2,3})\s*-\s*(\d{4})')
movil_z = movil_z.apply(
    lambda x: f"Z{x[0]}{x[1]}" if pd.notna(x[0]) else np.nan,
    axis=1
)

# 2️⃣ Buscar TODOS los patrones tipo 50-4025 (pero NO fechas)
def extraer_sin_z(texto):
    import re
    
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto).upper()
    
    # encontrar todos los patrones tipo 50-4025
    matches = re.findall(r'\b(\d{2,3})\s*-\s*(\d{4})\b', texto)
    
    for m in matches:
        # evitar fechas (ej: 03-04-2026 → el segundo número sería 2026 pero el primero es 03 → válido, pero queremos evitarlo)
        # regla: móviles suelen ser >= 50 en la primera parte
        if int(m[0]) >= 50:
            return f"{m[0]}{m[1]}"
    
    return np.nan

movil_sin_z = col.apply(extraer_sin_z)

# 3️⃣ números directos
numeros = col.str.extract(r'\b(\d{6})\b')[0]

# 🔥 unificar
notas['Movil_limpio'] = movil_z.fillna(movil_sin_z).fillna(numeros)

# resultado
notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos,estado_nota,Movil_limpio
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123251
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123260
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123262
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123513
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123516


In [14]:
col = notas['Observaciones'].astype(str).str.upper()

# Caso con Z → Z50-2025 → 502025
movil_z = col.str.extract(r'Z\s*\|?\s*(\d{2,3})\s*-\s*(\d{4})')
movil_z = movil_z.apply(
    lambda x: f"{x[0]}{x[1]}" if pd.notna(x[0]) else np.nan,
    axis=1
)

# Caso sin Z
def extraer_sin_z(texto):
    import re
    
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto).upper()
    
    matches = re.findall(r'\b(\d{2,3})\s*-\s*(\d{4})\b', texto)
    
    for m in matches:
        if int(m[0]) >= 50:
            return f"{m[0]}{m[1]}"
    
    return np.nan

movil_sin_z = col.apply(extraer_sin_z)

# números directos
numeros = col.str.extract(r'\b(\d{6})\b')[0]

# 🔥 unificar todo SIN Z
notas['Movil_limpio'] = movil_z.fillna(movil_sin_z).fillna(numeros)

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos,estado_nota,Movil_limpio
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123251
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123260
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123262
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123513
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,123516


In [15]:
# asegurar que sea string
col = notas['Movil_limpio'].astype(str)

notas['Movil_limpio'] = (
    col
    .where(col.str.startswith(('50', '52')), '0')  # deja solo 50 o 52
    .replace('nan', '0')                           # por si hay nan como texto
    .fillna('0')                                   # por si hay NaN reales
    .astype(int)                                   # convertir a entero
)

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos,estado_nota,Movil_limpio
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0


In [16]:
condiciones = [
    (notas['N° Vehículo'] == 0) & (notas['Movil_limpio'] == 0),
    (notas['N° Vehículo'] != 0) & (notas['N° Vehículo'] == notas['Movil_limpio'])
]

opciones = [
    'No considerar',
    'Coincide'
]

notas['Estado_bus'] = np.select(condiciones, opciones, default='No coincide')

notas.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por,cantidad_campos,estado_nota,Movil_limpio,Estado_bus
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0,No considerar
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0,No considerar
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0,No considerar
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123513/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0,No considerar
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO,14,ok,0,No considerar


In [17]:
def parse_observacion(texto):
    if pd.isna(texto):
        return {}

    pares = re.findall(r'([^:/]+):\s*([^/]+)', texto)
    return {k.strip(): v.strip() for k, v in pares}

df_obs = notas["Observaciones"].apply(parse_observacion).apply(pd.Series)

df_obs.head(3)


,ID,03,Motivo,Desde,Sentido,Descripción de desvío,Reportado por Regulador de CCZ,Reportado a TS,ticket,Reportado por,...,"Se crea tabla por introducir coche a ruta 614 tabla 20 Viaje 3, quedando como Tabla 45, Viaje 1, Móvil Z50-4205, operador 509648 . Reportado por",Empalme Turno 12  Supervisión San Cristóbal Fecha,1 Vueltas perdidas; Ruta 16-5 Villa Amalia; Tabla 14; Viaje 3; Por Falta de móvil; Móvil ; Operador ; De 17,Siendo las 17,Nota informativa ( Pison de emergencia se activa constantemente ) ticket,1 Vueltas eliminadas; Ruta 16-5 Villa Amalia; Tabla 2; Viaje 12; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3
0,123251,00,Mal estado de la Malla vial,052A08: Hasta: 429A08,Occidente - Oriente,Tomar avenida de las americas al oriente hasta...,Esteban Camilo Medina,William Varela,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,123260,00,Paso reducido,088A13: Hasta: 175A13,Norte - Sur,En diagonal 43a sur tomar la carrera 6b este a...,Esteban Camilo Medina,William Varela,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,123262,00,Paso reducido,173A13: Hasta: 089A13,Sur-Norte,En carrera 7a tomar la calle 44 sur al occiden...,Esteban Camilo Medina,William Varela,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
def parse_observacion(texto):
    if pd.isna(texto):
        return {}

    pares = re.findall(r'([^:/]+):\s*([^/]+)', texto)
    return {k.strip(): v.strip() for k, v in pares}


# Parsear observaciones
df_obs = notas["Observaciones"].apply(parse_observacion).apply(pd.Series)

# Agregar columnas base
df_obs = pd.concat(
    [notas[["Fecha","Id de línea","Nombre Línea","ID Nota","N° Vehículo","Código Conductor", "Tipo Nota", "Subtipo Nota", "Subsubtipo Nota", "Observaciones", "Creado Por","cantidad_campos","estado_nota","Movil_limpio","Estado_bus"]], df_obs],
    axis=1
)

df_obs.head(3)

,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,...,"Se crea tabla por introducir coche a ruta 614 tabla 20 Viaje 3, quedando como Tabla 45, Viaje 1, Móvil Z50-4205, operador 509648 . Reportado por",Empalme Turno 12  Supervisión San Cristóbal Fecha,1 Vueltas perdidas; Ruta 16-5 Villa Amalia; Tabla 14; Viaje 3; Por Falta de móvil; Móvil ; Operador ; De 17,Siendo las 17,Nota informativa ( Pison de emergencia se activa constantemente ) ticket,1 Vueltas eliminadas; Ruta 16-5 Villa Amalia; Tabla 2; Viaje 12; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3
0,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1/05/2026 2:09,10184,740,993893,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1/05/2026 2:09,10184,740,993895,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df_obs = df_obs.sort_values(by="ID", key=lambda x: x.isna())

df_obs.head(3)

,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,...,"Se crea tabla por introducir coche a ruta 614 tabla 20 Viaje 3, quedando como Tabla 45, Viaje 1, Móvil Z50-4205, operador 509648 . Reportado por",Empalme Turno 12  Supervisión San Cristóbal Fecha,1 Vueltas perdidas; Ruta 16-5 Villa Amalia; Tabla 14; Viaje 3; Por Falta de móvil; Móvil ; Operador ; De 17,Siendo las 17,Nota informativa ( Pison de emergencia se activa constantemente ) ticket,1 Vueltas eliminadas; Ruta 16-5 Villa Amalia; Tabla 2; Viaje 12; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3
0,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1686,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 90521/ 12-05-2026/ 03...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1687,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 122172/ 12-05-2026/ 0...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_obs.insert(
    0,
    "tipo_evento",
    np.where(df_obs["ID"].notna() & (df_obs["ID"] != ""), "Desvios", "Novedades de ruta")
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,"Se crea tabla por introducir coche a ruta 614 tabla 20 Viaje 3, quedando como Tabla 45, Viaje 1, Móvil Z50-4205, operador 509648 . Reportado por",Empalme Turno 12  Supervisión San Cristóbal Fecha,1 Vueltas perdidas; Ruta 16-5 Villa Amalia; Tabla 14; Viaje 3; Por Falta de móvil; Móvil ; Operador ; De 17,Siendo las 17,Nota informativa ( Pison de emergencia se activa constantemente ) ticket,1 Vueltas eliminadas; Ruta 16-5 Villa Amalia; Tabla 2; Viaje 12; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1686,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1687,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df_obs.columns = df_obs.columns.str.strip()

In [22]:
#Limpiar texto
mask = df_obs["tipo_evento"] == "Desvios"

extraidos = (
    df_obs.loc[mask, "Desde"]
    .astype(str)
    .str.findall(r"\d+[A-Z]\d+")
)

df_obs.loc[mask, "Parada_ini"] = extraidos.str[0]
df_obs.loc[mask, "Parada_fin"] = extraidos.str[1]

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,1 Vueltas perdidas; Ruta 16-5 Villa Amalia; Tabla 14; Viaje 3; Por Falta de móvil; Móvil ; Operador ; De 17,Siendo las 17,Nota informativa ( Pison de emergencia se activa constantemente ) ticket,1 Vueltas eliminadas; Ruta 16-5 Villa Amalia; Tabla 2; Viaje 12; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3,Parada_ini,Parada_fin
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,052A08,429A08
1686,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,162C08,114B09
1687,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,050A00,227B00


In [23]:
print(df_obs["Paradas"].str.contains("Entre", case=False, na=False).sum())

376


In [24]:
# Limpieza base
df_obs["Paradas"] = (
    df_obs["Paradas"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
)

# Extracción robusta
extraido = df_obs["Paradas"].str.extract(
    r'Entre\s*(.*?)\s*-\s*Hasta\s*(.*)',
    flags=re.IGNORECASE
)

# Aplicar solo si es Novedades de ruta
mask = df_obs["tipo_evento"].str.contains("Novedades", case=False, na=False)

df_obs.loc[mask, "parada_ini_afectada"] = extraido[0]
df_obs.loc[mask, "parada_fin_afectada"] = extraido[1]

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,Nota informativa ( Pison de emergencia se activa constantemente ) ticket,1 Vueltas eliminadas; Ruta 16-5 Villa Amalia; Tabla 2; Viaje 12; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3,Parada_ini,Parada_fin,parada_ini_afectada,parada_fin_afectada
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,052A08,429A08,NaN,NaN
1686,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,162C08,114B09,NaN,NaN
1687,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,050A00,227B00,NaN,NaN


In [25]:
# Regex para extraer código de parada
patron = r'(\d{3}[A-Za-z]\d{2})'

df_obs["parada_ini_afec"] = df_obs["parada_ini_afectada"].str.extract(patron)
df_obs["parada_fin_afec"] = df_obs["parada_fin_afectada"].str.extract(patron)

df_obs["parada_ini_afec"] = df_obs["parada_ini_afec"].str.upper()
df_obs["parada_fin_afec"] = df_obs["parada_fin_afec"].str.upper()

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,1 Vueltas adicionales; Ruta 16-5 Villa Amalia; Tabla 15; Viaje 1; Por Retoma; Móvil Z50-7059; Operador 505209; De 17,3,sentido,Empalme turno 24 de 3,Parada_ini,Parada_fin,parada_ini_afectada,parada_fin_afectada,parada_ini_afec,parada_fin_afec
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,052A08,429A08,NaN,NaN,NaN,NaN
1686,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,162C08,114B09,NaN,NaN,NaN,NaN
1687,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,050A00,227B00,NaN,NaN,NaN,NaN


In [26]:
#cruzar datos de paradas con paradas de la bitácora

maestro_paradas = paradas[[
    "cenefa",
    "nombre",
    "via",
    "direccion_bandera",
    "localidad",
    "longitud",
    "latitud"
]]

maestro_paradas.head()

,cenefa,nombre,via,direccion_bandera,localidad,longitud,latitud
0,001A00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481
1,001A01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091
2,001A02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867
3,001A03,Br. Julio Flórez,AC 100,AC 100 - KR 66A,Suba,-74.071439,4.689504
4,001A04,Avenida Calle 80,AK 68,AK 68 - CL 79D,Barrios Unidos,-74.080392,4.682626


In [27]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_ini"),
    left_on="Parada_ini",
    right_on="cenefa_ini",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,parada_fin_afectada,parada_ini_afec,parada_fin_afec,cenefa_ini,nombre_ini,via_ini,direccion_bandera_ini,localidad_ini,longitud_ini,latitud_ini
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,052A08,Estación Biblioteca Tintal,AV. Américas,AV. Américas - KR 82,Kennedy,-74.155502,4.638358
1,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,162C08,Br. Provivienda Oriental,AK 68,AK 68 - CL 22 Sur,Kennedy,-74.128606,4.611281
2,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,050A00,Br. Colombia,Av. Chile,AC 72 - KR 22,Barrios Unidos,-74.066179,4.662240


In [28]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_fin"),
    left_on="Parada_fin",
    right_on="cenefa_fin",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,localidad_ini,longitud_ini,latitud_ini,cenefa_fin,nombre_fin,via_fin,direccion_bandera_fin,localidad_fin,longitud_fin,latitud_fin
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,Kennedy,-74.155502,4.638358,429A08,Unidad Francisco José de Caldas,CL 26 Sur,CL 26 Sur - KR 79F,Kennedy,-74.153543,4.630447
1,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,Kennedy,-74.128606,4.611281,114B09,Br. Alquería La Fragua,AK 68,AK 68 - CL 35A Sur,Kennedy,-74.131334,4.605706
2,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,Barrios Unidos,-74.066179,4.662240,227B00,Clínica de Traumatología y Ortopedia,AK 11,AK 11 - CL 71,Chapinero,-74.059362,4.656432


In [29]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_ini_afec"),
    left_on="parada_ini_afec",
    right_on="cenefa_ini_afec",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,localidad_fin,longitud_fin,latitud_fin,cenefa_ini_afec,nombre_ini_afec,via_ini_afec,direccion_bandera_ini_afec,localidad_ini_afec,longitud_ini_afec,latitud_ini_afec
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,Kennedy,-74.153543,4.630447,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,Kennedy,-74.131334,4.605706,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,Chapinero,-74.059362,4.656432,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_fin_afec"),
    left_on="parada_fin_afec",
    right_on="cenefa_fin_afec",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,localidad_ini_afec,longitud_ini_afec,latitud_ini_afec,cenefa_fin_afec,nombre_fin_afec,via_fin_afec,direccion_bandera_fin_afec,localidad_fin_afec,longitud_fin_afec,latitud_fin_afec
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
columnas_cero = [
    "longitud_fin_afec",
    "latitud_fin_afec",
    "longitud_ini_afec",
    "latitud_ini_afec",
    "latitud_ini",
    "longitud_ini",
    "latitud_fin",
    "longitud_fin"
]

df_obs[columnas_cero] = df_obs[columnas_cero].fillna(0)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,N° Vehículo,Código Conductor,Tipo Nota,Subtipo Nota,Subsubtipo Nota,...,localidad_ini_afec,longitud_ini_afec,latitud_ini_afec,cenefa_fin_afec,nombre_fin_afec,via_fin_afec,direccion_bandera_fin_afec,localidad_fin_afec,longitud_fin_afec,latitud_fin_afec
0,Desvios,1/05/2026 2:09,10184,740,993891,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,Desvios,12/05/2026 1:52,10310,C101,1030420,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,Desvios,12/05/2026 1:52,10232,359,1030422,0,0,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0


In [32]:
print(df_obs['latitud_ini_afec'].unique())

[0.         4.595672   4.711096   4.71458849 4.712726   4.61736456
 4.611259   4.68726604 4.71237612 4.70989684 4.66641683 4.66404316
 4.66224012 4.65431866 4.55912767 4.55783355 4.55518421 4.591332
 4.64216871 4.58818432 4.63771138 4.696502   4.64883521 4.61788987
 4.62717405 4.65370423 4.70902773 4.61679569 4.64553868 4.6492986
 4.69172054 4.6783811  4.65751695 4.710687   4.66541111 4.578969
 4.63034102 4.63144653 4.71140535 4.71453737 4.65332272 4.62713431
 4.54690533 4.61127847 4.5958396  4.763744   4.56443772 4.49894324
 4.64366426 4.66725267 4.64713112 4.69823578 4.68360981 4.71532689
 4.719444   4.76521887 4.68086208 4.5693859  4.717202   4.72209518
 4.59186761 4.61058747 4.57670201 4.57219126 4.63579724 4.672921
 4.520352   4.61614404 4.530428   4.58789498 4.67100241 4.687194
 4.64105673 4.70371777 4.70938061 4.69140716 4.64195614 4.55822205
 4.61308045 4.64550224 4.62182918 4.56031582 4.55533669 4.5804361
 4.64225362 4.6854374  4.58221416 4.68892244 4.51693088 4.6049723
 4.572

In [33]:
df_obs.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Rutas_puntos_críticos/seguimiento_notas_{mes}.csv', sep=';', index=False)

In [34]:
base_notas.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Rutas_puntos_críticos/base_notas.csv', sep=';', index=False)